# Tahap 3: Ekstraksi Teks dari PDF

Notebook ini mengekstrak teks dari file PDF putusan menggunakan `pdfplumber` (dengan fallback ke `PyPDF2`).

- **Input:** `../data/raw/pdf/*.pdf` (hasil Tahap 2)
- **Output:** `../data/processed/teks_ekstrak/*.txt`

Setiap file `.txt` menyertakan header metadata (nomor, tanggal, kategori) diikuti teks lengkap putusan.

In [1]:
!pip install pdfplumber PyPDF2 pandas tqdm -q
print('Instalasi selesai!')

Instalasi selesai!


'pip' is not recognized as an internal or external command,
operable program or batch file.


## Konfigurasi

In [2]:
import os
import re
import glob
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime

PDF_DIR     = '../data/raw/pdf'
TXT_DIR     = '../data/processed/teks_ekstrak'
METADATA_DIR = '../data/raw/metadata'
os.makedirs(TXT_DIR, exist_ok=True)

# Jumlah halaman PDF maksimal yang diekstrak per file (hemat memori)
MAX_HALAMAN_PDF = 15

print('Konfigurasi dimuat.')
print('Input  PDF : ' + os.path.abspath(PDF_DIR))
print('Output TXT : ' + os.path.abspath(TXT_DIR))

Konfigurasi dimuat.
Input  PDF : c:\cbrs\data\raw\pdf
Output TXT : c:\cbrs\data\processed\teks_ekstrak


## Muat Metadata (Opsional)

Metadata digunakan untuk menambahkan header informasi pada setiap file teks.

In [3]:
# Coba muat metadata CSV untuk memperkaya header file teks
csv_files = sorted(glob.glob(os.path.join(METADATA_DIR, 'metadata_*.csv')))
df_meta = None
meta_lookup = {}

if csv_files:
    df_meta = pd.read_csv(csv_files[-1], encoding='utf-8-sig')
    # Buat lookup berdasarkan nama file PDF yang sesuai
    for _, row in df_meta.iterrows():
        nomor = str(row.get('nomor', ''))
        if nomor:
            kunci = re.sub(r'[<>:"/\\|?*\s]+', '_', nomor)[:60]
            meta_lookup[kunci] = row.to_dict()
    print('Metadata dimuat: ' + str(len(df_meta)) + ' entri dari ' + os.path.basename(csv_files[-1]))
else:
    print('Metadata tidak ditemukan — teks tetap diekstrak tanpa header lengkap.')

Metadata tidak ditemukan — teks tetap diekstrak tanpa header lengkap.


## Fungsi Ekstraksi Teks

In [4]:
def ekstrak_teks_pdf(path_pdf, max_halaman=MAX_HALAMAN_PDF):
    teks = ''
    try:
        import pdfplumber
        with pdfplumber.open(path_pdf) as pdf:
            for hal in pdf.pages[:max_halaman]:
                t = hal.extract_text()
                if t:
                    teks += t + '\n'
        if teks.strip():
            return teks.strip(), 'pdfplumber'
    except Exception:
        pass

    try:
        import PyPDF2
        with open(path_pdf, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for hal in reader.pages[:max_halaman]:
                teks += (hal.extract_text() or '') + '\n'
        return teks.strip(), 'PyPDF2'
    except Exception as e:
        return '[Gagal ekstrak: ' + str(e) + ']', 'error'


def tulis_teks_dengan_header(path_txt, teks, meta):
    with open(path_txt, 'w', encoding='utf-8') as f:
        f.write('NOMOR PUTUSAN : ' + str(meta.get('nomor', '')) + '\n')
        f.write('TANGGAL       : ' + str(meta.get('tanggal', '')) + '\n')
        f.write('PENGADILAN    : ' + str(meta.get('pengadilan', '')) + '\n')
        f.write('PARA PIHAK    : ' + str(meta.get('para_pihak', '')) + '\n')
        f.write('KATEGORI      : ' + str(meta.get('kategori', '')) + '\n')
        f.write('KLASIFIKASI   : ' + str(meta.get('klasifikasi', '')) + '\n')
        f.write('URL           : ' + str(meta.get('url_detail', '')) + '\n')
        f.write('=' * 60 + '\n')
        f.write(teks)


print('Fungsi ekstraksi siap.')

Fungsi ekstraksi siap.


## Jalankan Ekstraksi

In [5]:
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, '*.pdf')))

if not pdf_files:
    print('Tidak ada file PDF di ' + PDF_DIR)
    print('Jalankan notebook 02_unduh_pdf.ipynb terlebih dahulu.')
else:
    print('=' * 60)
    print('  EKSTRAKSI TEKS DARI ' + str(len(pdf_files)) + ' FILE PDF')
    print('=' * 60)

    berhasil, gagal, kosong = 0, 0, 0
    log = []

    for path_pdf in tqdm(pdf_files, desc='Ekstrak teks'):
        nama_pdf = os.path.basename(path_pdf)
        nama_base = os.path.splitext(nama_pdf)[0]
        path_txt = os.path.join(TXT_DIR, nama_base + '.txt')

        if os.path.exists(path_txt):
            berhasil += 1
            log.append({'file': nama_pdf, 'status': 'sudah_ada', 'panjang': os.path.getsize(path_txt)})
            continue

        teks, metode = ekstrak_teks_pdf(path_pdf)

        if not teks or teks.startswith('[Gagal'):
            print('  Gagal: ' + nama_pdf)
            gagal += 1
            log.append({'file': nama_pdf, 'status': 'gagal', 'panjang': 0})
            continue

        if len(teks) < 50:
            kosong += 1
            log.append({'file': nama_pdf, 'status': 'kosong', 'panjang': len(teks)})
            continue

        meta = meta_lookup.get(nama_base, {})
        tulis_teks_dengan_header(path_txt, teks, meta)
        berhasil += 1
        log.append({'file': nama_pdf, 'status': 'berhasil', 'metode': metode, 'panjang': len(teks)})

    print()
    print('=' * 60)
    print('  Berhasil : ' + str(berhasil) + ' file')
    print('  Gagal    : ' + str(gagal) + ' file')
    print('  Kosong   : ' + str(kosong) + ' file')
    print('  Output   : ' + os.path.abspath(TXT_DIR))
    print('=' * 60)

    df_log = pd.DataFrame(log)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    df_log.to_csv(os.path.join(METADATA_DIR, 'log_ekstrak_' + ts + '.csv'), index=False, encoding='utf-8-sig')
    display(df_log.head(10))

  EKSTRAKSI TEKS DARI 55 FILE PDF


Ekstrak teks:   0%|          | 0/55 [00:00<?, ?it/s]


  Berhasil : 55 file
  Gagal    : 0 file
  Kosong   : 0 file
  Output   : c:\cbrs\data\processed\teks_ekstrak


,file,status,metode,panjang
0,putusan_1005_pdt.p_2025_pn_tng_20260627213325.pdf,berhasil,pdfplumber,16811
1,putusan_114_pdt.p_2026_pn_mpw_20260627210712.pdf,berhasil,pdfplumber,11024
2,putusan_120_pdt.p_2026_pn_rbi_20260627210225.pdf,berhasil,pdfplumber,39999
3,putusan_122_pdt.p_2026_pn_rbi_20260627210319.pdf,berhasil,pdfplumber,39999
4,putusan_128_pdt.p_2026_pn_rbi_20260627210328.pdf,berhasil,pdfplumber,37405
5,putusan_131_pdt.p_2026_pn_rbi_20260627210109.pdf,berhasil,pdfplumber,30244
6,putusan_139_pdt.p_2026_pn_mpw_20260627210627.pdf,berhasil,pdfplumber,37096
7,putusan_141_pdt.p_2026_pn_mpw_20260627210444.pdf,berhasil,pdfplumber,41603
8,putusan_142_pdt.p_2026_pn_jmr_20260627213433.pdf,berhasil,pdfplumber,23668
9,putusan_142_pdt.p_2026_pn_mpw_20260627210930.pdf,berhasil,pdfplumber,46114
